# Fairness Metrics for Predictive Models

## Requirements

In [27]:
from aif360.datasets import GermanDataset
from aif360.sklearn.metrics import statistical_parity_difference, disparate_impact_ratio, equal_opportunity_difference

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

import pandas as pd 
import numpy as np

from PyFairnessAI.data import privileged_groups_sens, unprivileged_groups_sens

## Initial elements

### Data


In [3]:
german_data_aif = GermanDataset(
        protected_attribute_names=['age'],            
        privileged_classes=[lambda x: x >= 25],      
        features_to_drop=['personal_status', 'sex'] 
    )

# Cambiar los labels de la respuesta de 2 a 0 (label 2 es desfavorable y 1 es favorable) 
german_data_aif.labels[german_data_aif.labels.ravel() == 2] =  german_data_aif.labels[german_data_aif.labels.ravel() == 2] - 2
german_data_aif.unfavorable_label = german_data_aif.unfavorable_label - 2

response_name = german_data_aif.label_names[0]
response_favorable_label = german_data_aif.favorable_label 
response_unfavorable_label = german_data_aif.unfavorable_label 
sens_variable_name = german_data_aif.protected_attribute_names[0]
sens_privileged_groups = [privileged_groups_sens(german_data_aif)]
sens_unprivileged_groups = [unprivileged_groups_sens(german_data_aif)]

print('response_name:', response_name)
print('response_favorable_label:', response_favorable_label)
print('response_unfavorable_label:', response_unfavorable_label)
print('sens_variable_name:', sens_variable_name)
print('sens_privileged_groups:', sens_privileged_groups)
print('sens_unprivileged_groups:', sens_unprivileged_groups)

response_name: credit
response_favorable_label: 1.0
response_unfavorable_label: 0.0
sens_variable_name: age
sens_privileged_groups: [{'age': 1}]
sens_unprivileged_groups: [{'age': 0}]


In [6]:
data_arr = np.concatenate([german_data_aif.labels, german_data_aif.features], axis=1)
data_col_names = [response_name] + german_data_aif.feature_names

german_data_pd = pd.DataFrame(data=data_arr, columns=data_col_names)
german_data_pd.head(6)

,credit,month,credit_amount,investment_as_income_percentage,residence_since,age,number_of_credits,people_liable_for,status=A11,status=A12,...,housing=A152,housing=A153,skill_level=A171,skill_level=A172,skill_level=A173,skill_level=A174,telephone=A191,telephone=A192,foreign_worker=A201,foreign_worker=A202
0,1.0,6.0,1169.0,4.0,4.0,1.0,2.0,1.0,1.0,0.0,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0
1,0.0,48.0,5951.0,2.0,2.0,0.0,1.0,1.0,0.0,1.0,...,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0
2,1.0,12.0,2096.0,2.0,3.0,1.0,1.0,2.0,0.0,0.0,...,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0
3,1.0,42.0,7882.0,2.0,4.0,1.0,1.0,2.0,1.0,0.0,...,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0
4,0.0,24.0,4870.0,3.0,4.0,1.0,2.0,2.0,1.0,0.0,...,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0
5,1.0,36.0,9055.0,2.0,4.0,1.0,1.0,2.0,0.0,0.0,...,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0



### Response and predictors


In [7]:
quant_predictors = ['month', 'credit_amount', 'investment_as_income_percentage',
                    'residence_since', 'number_of_credits', 'people_liable_for']
cat_predictors = [x for x in german_data_pd.columns if x not in quant_predictors + [response_name]]
predictors = quant_predictors + cat_predictors

In [8]:
Y = german_data_pd[response_name]
X = german_data_pd[german_data_aif.feature_names] # X = german_data_pd[quant_predictors + cat_predictors]
A = german_data_pd[sens_variable_name] # sensitive variable / protected attribute


### Outer evaluation method: simple-validation (train-test split)

In [9]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, train_size=0.85, random_state=123, stratify=Y)

### Fairness metrics


In [10]:
logistic_reg = LogisticRegression(solver='liblinear', random_state=123)
logistic_reg.fit(X_train, Y_train)
Y_test_hat = logistic_reg.predict(X_test)
A_test = X_test[sens_variable_name] # sensitive variable / protected attribute in test set


#### Statistical parity difference


In [16]:
value = statistical_parity_difference(y_true=Y_test,
                                      y_pred=Y_test_hat, 
                                      prot_attr=A_test,
                                      priv_group=sens_privileged_groups[0][sens_variable_name], 
                                      pos_label=response_favorable_label)
value = float(value)
value

-0.32539682539682535

`statistical_parity_difference`  $\hspace{0.05cm}=\hspace{0.05cm} P\left(\widehat{Y} = \text{positive\_label} \hspace{0.1cm}|\hspace{0.1cm} A = \text{unprivileged}\right) - P\left(\widehat{Y} = \text{positive\_label} \hspace{0.1cm}|\hspace{0.1cm} A = \text{privileged}\right)$

**Properties**

- $\in [-1, 1]$

- The closer to $-1$, the more favorable are the results (predictions) of the models to the privileged class of the sensitive variable.

- The closer to $1$, the more favorable are the results (predictions) of the models to the unprivileged class of the sensitive variable.

- The closer to $0$, the more fairness (parity) in the model results (predictions).


**Doubts**

- No veo que esta metrica sea una medida de justicia en todos los casos. Ejemplo: concesion de creditos, variable sensible color de piel (blanco, negro). Imaginemos que el grupo de los negros reune una serie de condiciones que les hace objetivamente mas riesgosos que los blancos (peor situacion economica, mayores deudas etc). Que en este caso el modelo conceda significativamente mas creditos a los blancos que a los negros no seria injusto, no? Puede setar basado en criterios economicos objetivos que estan asociados a esos grupos, y no por el color de piel en si, no??


---


#### Absolute statistical parity difference


In [17]:
def abs_statistical_parity_difference(y_true, y_pred, prot_attr, priv_group, pos_label):

    value = statistical_parity_difference(y_true=y_true, y_pred=y_pred, prot_attr=prot_attr, priv_group=priv_group, pos_label=pos_label)
    return float(np.abs(value))

In [18]:
value = abs_statistical_parity_difference(y_true=Y_test,
                                          y_pred=Y_test_hat, 
                                          prot_attr=A_test,
                                          priv_group=sens_privileged_groups[0][sens_variable_name], 
                                          pos_label=response_favorable_label)
value

0.32539682539682535

`abs_statistical_parity_difference`  $\hspace{0.05cm}=\hspace{0.05cm} \left| P\left(\widehat{Y} = \text{positive\_label} \hspace{0.1cm}|\hspace{0.1cm} A = \text{unprivileged}\right) - P\left(\widehat{Y} = \text{positive\_label} \hspace{0.1cm}|\hspace{0.1cm} A = \text{privileged}\right) \right|$

- $P(\hat{Y} = \text{positive\_label} | A = \text{unprivileged})= \text{Proportion of credit granted to black people}$

- $P(\hat{Y} = \text{positive\_label} | A = \text{privileged})= \text{Proportion of credit granted to white people}$

**Properties**

- $\in [0,1]$
- The **closer to $0$** the less parity difference between the sensitive groups (**more fairness** (?)).
- The **closer to $1$** the more parity difference between the sensitive groups (**less fairness** (?)).


---


#### Disparate impact ratio


In [22]:
value = disparate_impact_ratio(y_true=Y_test, y_pred=Y_test_hat, prot_attr=A_test,
                               priv_group=sens_privileged_groups[0][sens_variable_name], 
                               pos_label=response_favorable_label)
value

0.6057692307692308

  `disparate_impact_ratio` $\hspace{0.05cm}=\hspace{0.05cm}\dfrac{P(\hat{Y} = \text{positive\_label} | A = \text{unprivileged})}
{Pr(\hat{Y} = \text{positive\_label} | A = \text{privileged})}$

- $P(\hat{Y} = \text{positive\_label} | A = \text{unprivileged})= \text{Proportion of credit granted to black people}$

- $P(\hat{Y} = \text{positive\_label} | A = \text{privileged})= \text{Proportion of credit granted to white people}$

**Properties**

- $\in [0,\infty]$
- The **closer to $1$**, the less parity difference between the sensitive groups (**more fairness**).
- The **closer to $0$ or to $\infty$** (**farther from $1$**) , the more parity difference between the sensitive groups (**less fairness**).

---

#### Absolute equal opportunity difference

In [ ]:
equal_opportunity_difference(y_true, y_pred, *, prot_attr=None, priv_group=1, pos_label=1, sample_weight=None)[source]


`equal_opportunity_difference` $\hspace{0.1cm}=\hspace{0.1cm} \left| \hspace{0.05cm} P( \hat{Y}=\text{positive\_label} \hspace{0.1cm}|\hspace{0.1cm} Y=\text{positive\_label}, A = \text{unprivilaged}) \hspace{0.1cm}-\hspace{0.1cm} P( \hat{Y}=\text{positive\_label} \hspace{0.1cm}|\hspace{0.1cm}  Y=\text{positive\_label},  A = \text{privilaged}) \hspace{0.05cm}\right| \\[0.5cm] \hspace{4.75cm} = \Big| \hspace{0.05cm} \text{True Positive Rate} \hspace{0.1cm}|\hspace{0.1cm} A=\text{unprivilaged} \hspace{0.1cm}-\hspace{0.1cm} \text{True Positive Rate} \hspace{0.1cm}|\hspace{0.1cm} A = \text{privilaged}\hspace{0.05cm} \Big|$

- $P( \hat{Y}=\text{positive\_label} \hspace{0.1cm}|\hspace{0.1cm} Y=\text{positive\_label}, A = \text{unprivilaged}) = \text{Proportion of credit granted by the model among black people that actually deserved it}$

- $P( \hat{Y}=\text{positive\_label} \hspace{0.1cm}|\hspace{0.1cm} Y=\text{positive\_label}, A = \text{unprivilaged}) = \text{Proportion of credit granted by the model among white people that actually deserved it}$

**Properties**

- $\in [0, 1]$
- The closer to $0$, the less difference of opportunity (more fairness).
- The closer to $1$, the more difference of opportunity (less fairness).


---